In [ ]:
!pip install -q dtreeviz graphviz cairosvg

In [ ]:
# Titanic Survival Prediction using Decision Tree
# Clean Google Colab version

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

import os
import contextlib

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

from dtreeviz import model as dtreeviz_model
import cairosvg

matplotlib.rcParams["font.family"] = "DejaVu Sans"


# ---------------------------------------------------------
# 1. Load dataset
# ---------------------------------------------------------

print("Loading dataset...")

df = pd.read_csv("train.csv")

print(f"Dataset loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")


# ---------------------------------------------------------
# 2. Exploratory Data Analysis
# ---------------------------------------------------------

print("Creating EDA plots...")

# Survival Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x="Survived", data=df)
plt.title("Survival Distribution")
plt.xlabel("Survived: 0 = Died, 1 = Survived")
plt.ylabel("Number of Passengers")
plt.savefig("survival_distribution.png", bbox_inches="tight", dpi=300)
plt.close()

# Survival by Sex
plt.figure(figsize=(6, 4))
sns.countplot(x="Sex", hue="Survived", data=df)
plt.title("Survival by Sex")
plt.xlabel("Sex")
plt.ylabel("Number of Passengers")
plt.legend(title="Survived", labels=["Died", "Survived"])
plt.savefig("survival_by_sex.png", bbox_inches="tight", dpi=300)
plt.close()

# Survival by Passenger Class
plt.figure(figsize=(6, 4))
sns.countplot(x="Pclass", hue="Survived", data=df)
plt.title("Survival by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Number of Passengers")
plt.legend(title="Survived", labels=["Died", "Survived"])
plt.savefig("survival_by_class.png", bbox_inches="tight", dpi=300)
plt.close()

# Age Distribution
plt.figure(figsize=(7, 4))
sns.histplot(df["Age"].dropna(), bins=30, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.savefig("age_distribution.png", bbox_inches="tight", dpi=300)
plt.close()

# Fare Distribution
plt.figure(figsize=(7, 4))
sns.histplot(df["Fare"], bins=30, kde=True)
plt.title("Fare Distribution")
plt.xlabel("Fare")
plt.ylabel("Frequency")
plt.savefig("fare_distribution.png", bbox_inches="tight", dpi=300)
plt.close()

print("EDA plots saved.")


# ---------------------------------------------------------
# 3. Data preprocessing
# ---------------------------------------------------------

print("Preprocessing data...")

df_processed = df.copy()

# Fill missing values
df_processed["Age"] = df_processed["Age"].fillna(df_processed["Age"].median())
df_processed["Embarked"] = df_processed["Embarked"].fillna(df_processed["Embarked"].mode()[0])

# Encode categorical columns
df_processed["Sex"] = df_processed["Sex"].map({
    "male": 0,
    "female": 1
})

df_processed["Embarked"] = df_processed["Embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})

# Remove unnecessary columns
df_processed = df_processed.drop(
    ["PassengerId", "Name", "Ticket", "Cabin"],
    axis=1
)

df_processed.to_csv("titanic_cleaned.csv", index=False)

print("Data preprocessing completed.")


# ---------------------------------------------------------
# 4. Feature and target selection
# ---------------------------------------------------------

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = df_processed[features]
y = df_processed["Survived"]


# ---------------------------------------------------------
# 5. Train/test split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


# ---------------------------------------------------------
# 6. Model training
# ---------------------------------------------------------

tree_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree_model.fit(X_train, y_train)

print("Decision Tree model trained.")


# ---------------------------------------------------------
# 7. Model evaluation
# ---------------------------------------------------------

y_pred = tree_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nModel Evaluation Results")
print("-" * 30)
print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-score:  {f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(cm)


# ---------------------------------------------------------
# 8. Save evaluation results
# ---------------------------------------------------------

with open("model_results.txt", "w", encoding="utf-8") as file:
    file.write("Titanic Survival Prediction using Decision Tree\n")
    file.write("=" * 50 + "\n\n")

    file.write(f"Dataset shape: {df.shape}\n")
    file.write(f"Training set size: {X_train.shape[0]}\n")
    file.write(f"Testing set size: {X_test.shape[0]}\n\n")

    file.write("Model Evaluation Metrics:\n")
    file.write(f"Accuracy:  {accuracy * 100:.2f}%\n")
    file.write(f"Precision: {precision * 100:.2f}%\n")
    file.write(f"Recall:    {recall * 100:.2f}%\n")
    file.write(f"F1-score:  {f1 * 100:.2f}%\n\n")

    file.write("Confusion Matrix:\n")
    file.write(str(cm))
    file.write("\n\n")

    file.write("Classification Report:\n")
    file.write(classification_report(
        y_test,
        y_pred,
        target_names=["Died", "Survived"]
    ))

print("Model results saved as model_results.txt")


# ---------------------------------------------------------
# 9. Confusion matrix visualization
# ---------------------------------------------------------

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=["Died", "Survived"],
    yticklabels=["Died", "Survived"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig("confusion_matrix.png", bbox_inches="tight", dpi=300)
plt.close()


# ---------------------------------------------------------
# 10. Feature importance visualization
# ---------------------------------------------------------

feature_importance = pd.Series(
    tree_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(
    x=feature_importance.values,
    y=feature_importance.index
)
plt.title("Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.savefig("feature_importance.png", bbox_inches="tight", dpi=300)
plt.close()

print("\nMost important feature:", feature_importance.index[0])


# ---------------------------------------------------------
# 11. Standard decision tree visualization
# ---------------------------------------------------------

plt.figure(figsize=(22, 12))
plot_tree(
    tree_model,
    feature_names=X.columns,
    class_names=["Died", "Survived"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Visualization")
plt.savefig("decision_tree.png", bbox_inches="tight", dpi=300)
plt.close()

print("Standard decision tree saved as decision_tree.png")


# ---------------------------------------------------------
# 12. dtreeviz decision tree visualization
# ---------------------------------------------------------

print("Creating dtreeviz visualization...")

with open(os.devnull, "w") as f, contextlib.redirect_stderr(f):
    viz_model = dtreeviz_model(
        tree_model,
        X_train=X_train,
        y_train=y_train,
        feature_names=features,
        target_name="Survived",
        class_names=["Died", "Survived"]
    )

    v = viz_model.view(
        scale=1.1,
        orientation="LR"
    )

    v.save("titanic_dtreeviz.svg")

    cairosvg.svg2png(
        url="titanic_dtreeviz.svg",
        write_to="titanic_dtreeviz.png",
        output_width=3000
    )

print("dtreeviz tree saved as titanic_dtreeviz.png")


# ---------------------------------------------------------
# 13. Final summary
# ---------------------------------------------------------

print("\nProject completed successfully.")
print("Generated files:")
print("- survival_distribution.png")
print("- survival_by_sex.png")
print("- survival_by_class.png")
print("- age_distribution.png")
print("- fare_distribution.png")
print("- confusion_matrix.png")
print("- feature_importance.png")
print("- decision_tree.png")
print("- titanic_dtreeviz.png")
print("- titanic_cleaned.csv")
print("- model_results.txt")

print(f"\nFinal accuracy: {accuracy * 100:.2f}%")